# Feature Engineering

The exploratory analysis revealed several features that are strongly associated with house prices, as well as groups of variables that describe related aspects of a property. In this notebook, we leverage this information to create new features that may help machine learning models capture important patterns in the data.

Feature engineering is the process of transforming existing variables into new representations that better reflect the underlying factors influencing the target variable. By combining related features, aggregating information, and creating meaningful indicators, we can often provide models with a more informative view of the problem than the raw data alone.

The features introduced in this notebook are motivated by simple domain knowledge about residential properties. In general, buyers tend to care about factors such as total living space, property age, overall quality, available amenities, and the amount of usable space dedicated to specific purposes. The engineered features below aim to capture these concepts more directly.

Before creating new features, we first load the processed dataset generated in the previous notebook. This dataset already includes the missing-value handling and data quality corrections identified during the exploratory data analysis stage.

We then import the libraries required for feature engineering and subsequent preprocessing steps.

In [68]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

# Load data processed and saved in the EDA notebook
df_full = pd.read_parquet(
    "processed_data/eda.parquet"
)

df_full

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,IsTrainSet
0,1,60,RL,65.0,8450,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,2,2008,WD,Normal,208500.0,True
1,2,20,RL,80.0,9600,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,5,2007,WD,Normal,181500.0,True
2,3,60,RL,68.0,11250,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,9,2008,WD,Normal,223500.0,True
3,4,70,RL,60.0,9550,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,2,2006,WD,Abnorml,140000.0,True
4,5,60,RL,84.0,14260,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,12,2008,WD,Normal,250000.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,6,2006,WD,Normal,NaN,False
2915,2916,160,RM,21.0,1894,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,4,2006,WD,Abnorml,NaN,False
2916,2917,20,RL,160.0,20000,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,9,2006,WD,Abnorml,NaN,False
2917,2918,85,RL,62.0,10441,Pave,None,Reg,Lvl,AllPub,...,None,MnPrv,Shed,700,7,2006,WD,Normal,NaN,False


## Engineered numerical features

### TotalRooms

The total number of rooms is often more informative than the individual room counts alone. This feature attempts to capture the overall functional capacity of the house.

In [69]:
df_full["TotalRooms"] = (
    df_full["TotRmsAbvGrd"]
    + df_full["KitchenAbvGr"]
)

### TotalBathrooms

The dataset distributes bathroom information across multiple variables. Combining them into a single measure provides a more complete representation of the property's sanitary facilities.

In [70]:
df_full["TotalBathrooms"] = (
    df_full["FullBath"]
    + 0.5*df_full["HalfBath"]
    + df_full["BsmtFullBath"]
    + 0.5*df_full["BsmtHalfBath"]
)

### HouseAge

The age of a property at the time of sale is likely to influence its market value. Newer houses often command higher prices due to modern construction standards, lower maintenance requirements, and updated amenities.

In [71]:
df_full["HouseAge"] = (
    df_full["YrSold"]
    - df_full["YearBuilt"]
)

### YearsSinceRemodel

The original remodeling year provides useful information, but the number of years since the last renovation may be more directly related to a property's perceived condition and market value.

In [72]:
df_full["YearsSinceRemodel"] = (
    df_full["YrSold"]
    - df_full["YearRemodAdd"]
)

### TotalArea

The original dataset stores living space across multiple variables corresponding to different levels of the house. While this information is useful, buyers are often interested in the total amount of usable space available. This feature aggregates the main living areas into a single measure of property size.

In [73]:
df_full["TotalArea"] = (
    df_full["TotalBsmtSF"]
    + df_full["1stFlrSF"]
    + df_full["2ndFlrSF"]
)

### GarageAge

A newer garage may be more desirable than an older one due to reduced wear and improved construction standards. This feature captures the age of the garage at the time of sale.

In [74]:
df_full["GarageAge"] = (
    df_full["YrSold"]
    - df_full["GarageYrBlt"]
)

### TotalPorchArea

Porches appear in several separate variables depending on their type. This feature aggregates them into a single measure of outdoor living space.

In [75]:
df_full["TotalPorchArea"] = (
    df_full["OpenPorchSF"]
    + df_full["EnclosedPorch"]
    + df_full["3SsnPorch"]
    + df_full["ScreenPorch"]
)

### QualityArea

Previous analysis showed that both living area and overall quality are strongly associated with sale price. This feature captures the interaction between these variables, allowing the model to distinguish between houses that are large, high-quality, or both. It can be interpreted as a rough measure of quality-adjusted living space.

In [76]:
df_full["QualityArea"] = (
    df_full["OverallQual"]
    * df_full["GrLivArea"]
)

### TotalQualityScore

The overall quality and overall condition ratings describe complementary aspects of a property. Combining them creates a broader measure of the house's general quality.

In [77]:
df_full["TotalQualityScore"] = (
    df_full["OverallQual"]
    + df_full["OverallCond"]
)

### LuxuryFeatureCount

This feature counts the number of selected amenities present in a property. While each amenity contributes information individually, combining them into a single feature provides a simple measure of the property's amenity level.

The term luxury is used loosely here to refer to features that are commonly associated with larger or more valuable homes. Higher values indicate that a property offers a greater number of these amenities and may therefore command a higher market price.

In [78]:
df_full["LuxuryFeatureCount"] = (
    (df_full["PoolArea"] > 0).astype(int)
    + (df_full["GarageArea"] > 0).astype(int)
    + (df_full["TotalBsmtSF"] > 0).astype(int)
    + (df_full["Fireplaces"] > 0).astype(int)
)

### GarageAreaPerCar

The total garage area and the number of vehicle spaces provide complementary information about the garage. By dividing the area by the number of parking spaces, we obtain a rough measure of the amount of space available per vehicle.

This feature serves as an example of a ratio-based transformation, which can sometimes reveal relationships that are not apparent from the original variables alone. Larger values may indicate more spacious garages with additional storage or workshop space.

When constructing this feature, care must be taken to avoid division by zero. Properties without a garage have `GarageCars = 0`, so the ratio is defined as 0 for these observations.

In [79]:
df_full["GarageAreaPerCar"] = np.where(
    df_full["GarageCars"] > 0,
    df_full["GarageArea"] / df_full["GarageCars"],
    0
)

### LivingAreaPerRoom

The total above-ground living area and the number of rooms both provide useful information about the size of a house. However, they describe different aspects of the property. Two houses may have the same living area but very different room layouts.

By dividing the living area by the number of rooms, we obtain a rough measure of the average room size. This ratio may help distinguish between houses with many small rooms and those with fewer, more spacious rooms.

As with `GarageAreaPerCar`, or any ratio-based feature, care must be taken to avoid division by zero. In this dataset, all properties have at least one room above ground, but we nevertheless define the ratio as 0 whenever TotRmsAbvGrd is zero for robustness.

In [80]:
df_full["LivingAreaPerRoom"] = np.where(
    df_full["TotRmsAbvGrd"] > 0,
    df_full["GrLivArea"] / df_full["TotRmsAbvGrd"],
    0
)

## Binary indicator features

Several variables naturally describe the presence or absence of a property characteristic. While these variables can be represented as boolean values, converting them to integers (`0` or `1`) provides a format that is directly compatible with most machine learning algorithms and simplifies subsequent preprocessing.

### IsRemodeled

Indicates whether the property has been remodeled since its original construction.

In [81]:
df_full["IsRemodeled"] = (
    df_full["YearBuilt"] != df_full["YearRemodAdd"]
).astype(int)

### HasPool

Indicates whether the property includes a swimming pool.

In [82]:
df_full["HasPool"] = (
    df_full["PoolArea"] > 0
).astype(int)

### HasGarage

Indicates whether the property includes a garage.

In [83]:
df_full["HasGarage"] = (
    df_full["GarageArea"] > 0
).astype(int)

### HasBasement

Indicates whether the property includes a basement.

In [84]:
df_full["HasBasement"] = (
    df_full["TotalBsmtSF"] > 0
).astype(int)

### HasFireplace

Indicates whether the property includes at least one fireplace.

In [85]:
df_full["HasFireplace"] = (
    df_full["Fireplaces"] > 0
).astype(int)

## Summary

In this notebook, we engineered a collection of new features designed to provide a more informative representation of the properties than the raw dataset alone.

Key additions include:

* Aggregate features such as `TotalArea`, `TotalBathrooms`, and `TotalPorchArea`, which combine related measurements into more comprehensive indicators.
* Time-based features such as `HouseAge`, `YearsSinceRemodel`, and `GarageAge`, which capture the age and renovation history of a property.
* Interaction and ratio features such as `QualityArea`, `GarageAreaPerCar`, and `LivingAreaPerRoom`, which describe relationships between existing variables.
* Binary indicator variables such as `HasGarage`, `HasPool`, and `IsRemodeled`, which explicitly represent the presence or absence of important property characteristics.
* The `LuxuryFeatureCount` feature, which summarizes several desirable amenities into a single score.

These engineered features incorporate domain knowledge and may help machine learning models capture patterns that are not immediately apparent from the original variables.

The dataset is now ready for the final preprocessing steps required for modeling, including the transformation of skewed variables and the encoding of categorical features. In the next notebook, we will prepare the feature matrix, train several regression models, evaluate their performance, and generate a Kaggle submission.

## Save the engineered dataset

The engineered features created in this notebook will be used throughout the modeling stage. To avoid repeating the feature engineering pipeline, we save the resulting dataset to disk.

The next notebook will load this dataset, perform the remaining preprocessing steps required by the models, and proceed with training and evaluation.

In [86]:
df_full.to_parquet(
    "processed_data/engineered.parquet",
    index=False
)